# VisTAD: Grand Model Training-Time Top-K Channel Selection Ablation (Kaggle GPU)

**Experimental Objectives:**
1. **Training-Time Gradient Routing Ablation:** Trains Grand Foundation Models (`UniversalFoundationTCN_Pro`) from scratch across $K_{\text{train}} \in \{1, 3, 5, 10, \text{'all'}\}$ on the multi-domain mixture (SWaT + SMD + GECCO + MSL).
2. **Strict Permutation-Equivariance:** Enforces random channel dropping (40% - 100%) and random channel permutation/shuffling per batch.
3. **Elimination of Cold-Start Starvation:** Causal receptive field prefix $L_{\text{prefix}} = 250$ steps ($L_{\text{RF}} - 1$).
4. **Continuous Streaming Evaluation:** Evaluates every model on held-out test slices for all 4 benchmarks (SWaT, SMD, GECCO, MSL) recording `<DATASET>_<METRIC>`.
5. **Artifact Persistence:** Exports all model checkpoints, epoch-by-epoch loss trajectories, and structured JSON summaries to `/kaggle/working/`.


In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/UniFoundation'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained Top-K training ablation suite codebase...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
    zf.extractall('.')
print('Codebase unpacked successfully.')
print(f'Extracted top-level files: {sorted(os.listdir("."))}')


In [ ]:
# --- Step 2: GPU Accelerator Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'Detected {n_gpus} GPU(s):')
    for i in range(n_gpus):
        prop = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {prop.name} | VRAM: {prop.total_memory / 1e9:.2f} GB')


In [ ]:
# --- Step 3: Multi-Dataset Auto-Discovery & Path Linking ---
import os, glob, shutil

for ds in ['swat', 'smd/test', 'smd/test_label', 'gecco', 'msl']:
    os.makedirs(os.path.join('data', ds), exist_ok=True)
os.makedirs('cache', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results/ablations', exist_ok=True)

# 1. SWaT
swat_files = glob.glob('/kaggle/input/**/SWaT_Dataset_*.csv', recursive=True) + glob.glob('/kaggle/input/**/swat/*.csv', recursive=True)
for p in set(swat_files):
    fname = os.path.basename(p)
    target = os.path.join('data/swat', fname)
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/swat"))} SWaT files.')

# 2. SMD
smd_test_files = glob.glob('/kaggle/input/**/smd/test/*.txt', recursive=True)
for p in smd_test_files:
    target = os.path.join('data/smd/test', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)

smd_label_files = glob.glob('/kaggle/input/**/smd/test_label/*.txt', recursive=True)
for p in smd_label_files:
    target = os.path.join('data/smd/test_label', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/smd/test"))} SMD test files.')

# 3. GECCO
gecco_files = glob.glob('/kaggle/input/**/gecco_water.csv', recursive=True) + glob.glob('/kaggle/input/**/gecco/*.csv', recursive=True)
for p in set(gecco_files):
    target = os.path.join('data/gecco', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/gecco"))} GECCO files.')

# 4. MSL
msl_files = glob.glob('/kaggle/input/**/msl_*.csv', recursive=True) + glob.glob('/kaggle/input/**/msl/*.csv', recursive=True)
for p in set(msl_files):
    target = os.path.join('data/msl', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/msl"))} MSL files.')

# 5. Offline VLM Teacher Caches
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') and 'cache' in f:
            src_p = os.path.join(root, f)
            dst_p = os.path.join('cache', f)
            if not os.path.exists(dst_p):
                try: os.symlink(src_p, dst_p)
                except Exception: shutil.copy2(src_p, dst_p)
print(f'Available caches: {sorted(os.listdir("cache"))}')


In [ ]:
# --- Step 4: Execute Grand Model Top-K Training Ablation Suite ---
import os, sys, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    sys.executable, 'run_top_k_training_ablation.py',
    '--data_dir', 'data',
    '--cache_dir', 'cache',
    '--k_vals', '1', '3', '5', '10', 'all',
    '--epochs', '10',
    '--batch_size', '32',
    '--prefix_len', '250',
    '--window_size', '200',
    '--checkpoint_dir', 'checkpoints',
    '--output_json', 'results/ablations/top_k_training_ablation_grand.json',
    '--output_report', 'results/ablations/TOP_K_TRAINING_ABLATION_REPORT.md'
]

print(f'Executing command: {" ".join(cmd)}')
t0 = time.time()
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
process.stdout.close()
rc = process.wait()
print(f'Top-K Training Ablation process exited with return code: {rc} (Elapsed: {(time.time() - t0)/60:.2f} mins)')
if rc != 0:
    raise RuntimeError(f'Top-K Training Ablation failed with return code {rc}')


In [ ]:
# --- Step 5: Export Checkpoints & Reports to /kaggle/working/ ---
import os, shutil

out_ckpt_dir = '/kaggle/working/checkpoints'
out_res_dir = '/kaggle/working/results'
os.makedirs(out_ckpt_dir, exist_ok=True)
os.makedirs(out_res_dir, exist_ok=True)

# Copy checkpoints
for f in os.listdir('checkpoints'):
    src_p = os.path.join('checkpoints', f)
    dst_p = os.path.join(out_ckpt_dir, f)
    shutil.copy2(src_p, dst_p)
    print(f'Exported checkpoint: {dst_p} ({os.path.getsize(dst_p)/1e6:.2f} MB)')

# Copy results
for root, dirs, files in os.walk('results'):
    for f in files:
        src_p = os.path.join(root, f)
        dst_p = os.path.join(out_res_dir, f)
        shutil.copy2(src_p, dst_p)
        print(f'Exported result artifact: {dst_p}')

report_path = 'results/ablations/TOP_K_TRAINING_ABLATION_REPORT.md'
if os.path.exists(report_path):
    print('\n' + '=' * 80)
    print('FINAL TOP-K TRAINING ABLATION REPORT:')
    print('=' * 80)
    with open(report_path, 'r') as rf:
        print(rf.read())
